In [1]:
try:
    ip = get_ipython()
    if ip is not None:
        ip.run_line_magic('load_ext', 'cudf.pandas')
        print("cudf.pandas GPU acceleration enabled.")
except Exception as e:
    print(f"cudf.pandas not available, continuing with standard pandas. ({e})")


cudf.pandas GPU acceleration enabled.


In [2]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [3]:
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"Is NVIDIA GPU available? {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Using GPU device: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected, running on CPU.")


PyTorch Version: 2.5.1
Is NVIDIA GPU available? True
Using GPU device: NVIDIA T400 4GB


In [4]:
def _read_forcing_file(f_file):
    with open(f_file, 'r') as fh:
        lines = fh.readlines()

    header_idx = None
    for i, line in enumerate(lines[:10]):
        if 'dayl' in line and 'prcp' in line:
            header_idx = i
            break
    if header_idx is None:
        header_idx = 3

    df_forcing = pd.read_csv(f_file, sep=r'\s+', skiprows=header_idx)
    df_forcing.columns = [str(c).strip() for c in df_forcing.columns]
    return df_forcing

In [ ]:
def load_camels_data(base_dir):
    """
    Loads and processes CAMELS-US data arrays with a live region progress meter.
    """
    print("Step 1: Processing Static Attributes...")
    attributes_path = os.path.join(base_dir, "camels_attributes_v2.0")
    attribute_files = glob.glob(os.path.join(attributes_path, "camels_*.txt"))
    
    if len(attribute_files) == 0:
        raise FileNotFoundError(f"[ERROR] No attribute files found in: {os.path.abspath(attributes_path)}")
    
    static_df_list = []
    for file in attribute_files:
        df = pd.read_csv(file, sep=';', dtype={'gauge_id': str})
        df['gauge_id'] = df['gauge_id'].str.strip().str.zfill(8)
        df.set_index('gauge_id', inplace=True)
        static_df_list.append(df)
        
    static_master_df = pd.concat(static_df_list, axis=1)
    static_master_df = static_master_df.loc[:, ~static_master_df.columns.duplicated()]
    static_master_df = static_master_df.select_dtypes(include=[np.number])
    static_master_df = static_master_df.fillna(static_master_df.mean())
    
    static_dict = {str(gauge).strip().zfill(8): static_master_df.loc[gauge].values for gauge in static_master_df.index}
    print(f"Successfully loaded static attributes for {len(static_dict)} basins.")

    print("\nStep 2: Processing Dynamic Forcings & Streamflow Targets...")
    forcing_dict = {}
    target_dict = {}
    
    forcing_base_path = os.path.join(base_dir, "basin_mean_forcing", "daymet")
    stream_base_path = os.path.join(base_dir, "usgs_streamflow") 
    
    all_weather_records = []
    basin_meta_tracker = {}

  
    for region_folder in range(1, 19):
        folder_str = f"{region_folder:02d}"
        region_forcing_dir = os.path.join(forcing_base_path, folder_str)
        
        if not os.path.exists(region_forcing_dir):
            continue
            

        print(f" -> Loading & parsing files for Region {folder_str}/18...", end="\r")
            
        forcing_files = glob.glob(os.path.join(region_forcing_dir, "*_forcing_leap.txt"))
        
        for f_file in forcing_files:
            filename = os.path.basename(f_file)
            gauge_id = filename.split('_')[0].strip().zfill(8)
            
            if gauge_id not in static_dict:
                continue
                
            df_forcing = pd.read_csv(f_file, sep=r'\s+', skiprows=3)
            weather_features = ['dayl(s)', 'prcp(mm/day)', 'srad(W/m2)', 'swe(mm)', 'tmax(C)', 'tmin(C)', 'vp(Pa)']
            weather_data = df_forcing[weather_features].values
            
            s_file_pattern = os.path.join(stream_base_path, folder_str, f"{gauge_id}_*.txt")
            s_files = glob.glob(s_file_pattern)
            
            if not s_files:
                continue
                
            df_stream = pd.read_csv(s_files[0], sep=r'\s+', header=None)
            flow_column = pd.to_numeric(df_stream.iloc[:, -1], errors='coerce')
            stream_flow = flow_column.values.reshape(-1, 1)
            
            min_len = min(len(weather_data), len(stream_flow))
            weather_data = weather_data[:min_len]
            stream_flow = stream_flow[:min_len]

            stream_flow[np.isnan(stream_flow) | (stream_flow < 0)] = 0.0
            
            all_weather_records.append(weather_data)
            basin_meta_tracker[gauge_id] = (weather_data, stream_flow)
            
    print("\nProcessing complete! Building global scales and tensors...")
    
    if len(all_weather_records) == 0:
        raise FileNotFoundError("[ERROR] Still could not match basin files. Double-check your regional 01-18 subfolders.")
            
    flat_weather = np.vstack(all_weather_records)
    scaler_weather = StandardScaler()
    scaler_weather.fit(flat_weather)
    
    for gauge_id, (w_data, s_flow) in basin_meta_tracker.items():
        forcing_dict[gauge_id] = scaler_weather.transform(w_data)
        target_dict[gauge_id] = s_flow
        
    print(f"Successfully compiled data arrays for {len(forcing_dict)} active basins.")
    return forcing_dict, static_dict, target_dict

In [ ]:
import torch
from torch.utils.data import Dataset

class CamelsSequenceDataset(Dataset):
    def __init__(self, basin_ids, forcing_dict, static_dict, streamflow_dict, window_len=365):
        self.window_len = window_len
        self.samples = []
        
        
        for b_id in basin_ids:
            forcings = forcing_dict[b_id]       
            statics = static_dict[b_id]         
            flow = streamflow_dict[b_id]         
            
            total_days = len(forcings)
           
            for t in range(0, total_days - window_len, 1):
                self.samples.append({
                    'dynamic': forcings[t : t + window_len],  
                    'static': statics,                       
                    'target': flow[t : t + window_len]        
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        return (
            torch.tensor(sample['dynamic'], dtype=torch.float32),
            torch.tensor(sample['static'], dtype=torch.float32),
            torch.tensor(sample['target'], dtype=torch.float32)
        )

In [ ]:
import torch.nn as nn

class BiLSTMEncoder(nn.Module):
    def __init__(self, dynamic_dim=7, static_dim=30, hidden_dim=128):
        super(BiLSTMEncoder, self).__init__()
        self.hidden_dim = hidden_dim
   
        self.state_projector = nn.Linear(static_dim, hidden_dim)
   
        fused_dim = dynamic_dim + static_dim
        self.bilstm = nn.LSTM(fused_dim, hidden_dim, batch_first=True, bidirectional=True)

    def forward(self, x_dynamic, x_static):
        batch_size = x_dynamic.size(0)
        
     
        seq_len = x_dynamic.size(1)
        x_static_tiled = x_static.unsqueeze(1).repeat(1, seq_len, 1) 
        x_fused = torch.cat([x_dynamic, x_static_tiled], dim=-1) 
        
        h0_dir = self.state_projector(x_static).unsqueeze(0) 
        c0_dir = self.state_projector(x_static).unsqueeze(0) 
        
        h0 = torch.cat([h0_dir, h0_dir], dim=0) 
        c0 = torch.cat([c0_dir, c0_dir], dim=0) 
        
      
        encoder_outputs, (hn, cn) = self.bilstm(x_fused, (h0, c0))
      
        return encoder_outputs

In [ ]:
class AttentionDecoderStep(nn.Module):
    def __init__(self, dynamic_dim=7, static_dim=30, hidden_dim=128):
        super(AttentionDecoderStep, self).__init__()
        self.hidden_dim = hidden_dim
        
        self.W_s = nn.Linear(hidden_dim, hidden_dim)
        self.W_h = nn.Linear(2 * hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1)
        
        decoder_input_dim = dynamic_dim + static_dim + (2 * hidden_dim)
        self.lstm_cell = nn.LSTMCell(decoder_input_dim, hidden_dim)
        self.regression_head = nn.Linear(hidden_dim, 1)

    def forward(self, x_dynamic_t, x_static, s_prev, c_prev, encoder_outputs):
       
        scores = self.v(torch.tanh(self.W_h(encoder_outputs) + self.W_s(s_prev).unsqueeze(1))) # (Batch, 365, 1)
        alphas = torch.softmax(scores, dim=1) 
       
        context = torch.sum(alphas * encoder_outputs, dim=1)
        
      
        cell_input = torch.cat([x_dynamic_t, x_static, context], dim=-1)
        s_next, c_next = self.lstm_cell(cell_input, (s_prev, c_prev))
        
       
        y_hat_t = self.regression_head(s_next) 
        return y_hat_t, s_next, c_next

In [ ]:
class FullSeq2SeqGenerator(nn.Module):
    def __init__(self, dynamic_dim=7, static_dim=30, hidden_dim=128):
        super(FullSeq2SeqGenerator, self).__init__()
        self.hidden_dim = hidden_dim
        self.encoder = BiLSTMEncoder(dynamic_dim, static_dim, hidden_dim)
        self.decoder_step = AttentionDecoderStep(dynamic_dim, static_dim, hidden_dim)
        
        self.decoder_init_layer = nn.Linear(static_dim, hidden_dim)

    def forward(self, x_dynamic, x_static):
        batch_size = x_dynamic.size(0)
        seq_len = x_dynamic.size(1) # 365
        
        encoder_outputs = self.encoder(x_dynamic, x_static) 
        
        
        s_t = torch.tanh(self.decoder_init_layer(x_static)) 
        c_t = torch.zeros(batch_size, self.hidden_dim, device=x_dynamic.device)
        
        outputs = []
        
        for t in range(seq_len):
            x_dynamic_t = x_dynamic[:, t, :] 
            
            y_hat_t, s_t, c_t = self.decoder_step(x_dynamic_t, x_static, s_t, c_t, encoder_outputs)
            outputs.append(y_hat_t)
            
        full_hydrograph = torch.stack(outputs, dim=1) 
        return full_hydrograph

In [ ]:
class FastSeq2SeqGenerator(nn.Module):
    def __init__(self, dynamic_dim=7, static_dim=30, hidden_dim=128):
        super(FastSeq2SeqGenerator, self).__init__()
        self.encoder = BiLSTMEncoder(dynamic_dim, static_dim, hidden_dim)
        self.regression_head = nn.Linear(2 * hidden_dim, 1)

    def forward(self, x_dynamic, x_static):
        encoder_outputs = self.encoder(x_dynamic, x_static)
        full_hydrograph = self.regression_head(encoder_outputs)  # (B, 365, 1)
        return full_hydrograph


In [ ]:
class CamelsUngaugedDataset(Dataset):
    def __init__(self, basin_ids, forcing_dict, static_dict, target_dict, window_len=365, step_size=10):

        self.samples = []
        for b_id in basin_ids:
            if b_id not in forcing_dict or b_id not in static_dict or b_id not in target_dict:
                continue
                
            x_dyn = forcing_dict[b_id]   
            x_stat = static_dict[b_id]   
            y_flow = target_dict[b_id]   
            
            total_days = x_dyn.shape[0]
            for t in range(0, total_days - window_len + 1, step_size):
                self.samples.append({
                    'dynamic': x_dyn[t : t + window_len],
                    'static': x_stat,
                    'target': y_flow[t : t + window_len]
                })
                
        print(f"Created dataset split containing {len(self.samples)} sequence windows.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        return (
            torch.tensor(sample['dynamic'], dtype=torch.float32),
            torch.tensor(sample['static'], dtype=torch.float32),
            torch.tensor(sample['target'], dtype=torch.float32)
        )

In [ ]:
def get_regional_basin_splits(base_dir, test_region_num=18):
    forcing_base_path = os.path.join(base_dir, "basin_mean_forcing", "daymet")
    
    train_basins = []
    test_basins = []
    
    test_folder_str = f"{test_region_num:02d}"
    
    for r in range(1, 19):
        folder_str = f"{r:02d}"
        region_dir = os.path.join(forcing_base_path, folder_str)
        
        if not os.path.exists(region_dir):
            continue
            
        forcing_files = glob.glob(os.path.join(region_dir, "*_forcing_leap.txt"))
        for f_file in forcing_files:
            gauge_id = os.path.basename(f_file).split('_')[0]
            
            if folder_str == test_folder_str:
                test_basins.append(gauge_id) 
            else:
                train_basins.append(gauge_id) 
                
    print(f"\nExperimental Split Configuration:")
    print(f" -> Gauged Basins (Training): {len(train_basins)} basins from regions 1-{test_region_num-1}")
    print(f" -> Ungauged Basins (Testing): {len(test_basins)} basins from hidden region {test_region_num}")
    
    return train_basins, test_basins

In [ ]:
class KGEWithLoss(nn.Module):
    def __init__(self):
        super(KGEWithLoss, self).__init__()

    def forward(self, y_pred, y_true):
        y_pred = y_pred.view(-1)
        y_true = y_true.view(-1)
        
        mean_pred = torch.mean(y_pred)
        mean_true = torch.mean(y_true)
        
        std_pred = torch.std(y_pred)
        std_true = torch.std(y_true)
        
        r = torch.mean((y_pred - mean_pred) * (y_true - mean_true)) / (std_pred * std_true + 1e-6)
        
        alpha = std_pred / (std_true + 1e-6)
        beta = mean_pred / (mean_true + 1e-6)
        
      
        kge = 1.0 - torch.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)
        
        return 1.0 - kge

In [ ]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler  

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True 

BASE_DIRECTORY = "./data" 

forcings, statics, targets = load_camels_data(BASE_DIRECTORY)


train_ids, test_ids = get_regional_basin_splits(BASE_DIRECTORY, test_region_num=18)

print("\n[Data] Generating Rolling Time-Windows...")
STEP_SIZE = 50   
train_dataset = CamelsUngaugedDataset(train_ids, forcings, statics, targets, window_len=365, step_size=STEP_SIZE)
test_dataset = CamelsUngaugedDataset(test_ids, forcings, statics, targets, window_len=365, step_size=STEP_SIZE)

BATCH_SIZE = 64 
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                           num_workers=4, pin_memory=True, persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
                          num_workers=4, pin_memory=True, persistent_workers=True)

sample_dyn, sample_stat, sample_target = next(iter(train_loader))
DYNAMIC_DIM = sample_dyn.shape[2]
STATIC_DIM = sample_stat.shape[1]
HIDDEN_DIM = 128    
EPOCHS = 10         
LEARNING_RATE = 0.001

print(f"\n>>> Dimensions Autodetected Successfully! <<<")
print(f"Weather Features Input Matrix: {DYNAMIC_DIM}")
print(f"Static Attributes Vector Length: {STATIC_DIM}")
print(f"Train windows: {len(train_dataset)} | Test windows: {len(test_dataset)} | Batch size: {BATCH_SIZE}")
print(f"Batches per epoch: {len(train_loader)} (was {767664 // 16} before the change)")

model = FastSeq2SeqGenerator(dynamic_dim=DYNAMIC_DIM, static_dim=STATIC_DIM, hidden_dim=HIDDEN_DIM).to(device)
criterion = KGEWithLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

scaler = GradScaler('cuda') 

print(f"Model successfully loaded onto {device}.")
print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


print("\n--- Starting Training Phase on Gauged Basins ---")
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    for batch_idx, (b_dynamic, b_static, b_target) in enumerate(train_loader):
        b_dynamic = b_dynamic.to(device, non_blocking=True)
        b_static = b_static.to(device, non_blocking=True)
        b_target = b_target.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast('cuda'): 
            generated_flow = model(b_dynamic, b_static)
            loss = criterion(generated_flow, b_target)

        scaler.scale(loss).backward()
        
        scaler.step(optimizer)
   
        scaler.update()

        epoch_loss += loss.item()

    avg_epoch_loss = epoch_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] Completed | Average Train Loss: {avg_epoch_loss:.4f} (Avg KGE: {1.0 - avg_epoch_loss:.4f})")

print("\n--- Starting Evaluation Phase on Hidden Ungauged Basins ---")
model.eval()

all_predictions = []
all_ground_truths = []

with torch.no_grad():
    for b_dynamic, b_static, b_target in test_loader:
        b_dynamic = b_dynamic.to(device, non_blocking=True)
        b_static = b_static.to(device, non_blocking=True)

        with autocast('cuda'): 
            generated_flow = model(b_dynamic, b_static)

        all_predictions.append(generated_flow.cpu().numpy().flatten())
        all_ground_truths.append(b_target.numpy().flatten())

predictions = np.concatenate(all_predictions)
ground_truths = np.concatenate(all_ground_truths)

def calculate_nse(pred, true):
    numerator = np.sum((true - pred) ** 2)
    denominator = np.sum((true - np.mean(true)) ** 2)
    return 1.0 - (numerator / (denominator + 1e-6))

def calculate_kge(pred, true):
    mean_pred, mean_true = np.mean(pred), np.mean(true)
    std_pred, std_true = np.std(pred), np.std(true)

    r = np.corrcoef(pred, true)[0, 1] if std_pred > 0 and std_true > 0 else 0
    alpha = std_pred / (std_true + 1e-6)
    beta = mean_pred / (mean_true + 1e-6)

    kge_score = 1.0 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)
    return kge_score

final_nse = calculate_nse(predictions, ground_truths)
final_kge = calculate_kge(predictions, ground_truths)


print(f"Validation Basin Region  : Region 18 (Hidden Ungauged)")
print(f"Nash-Sutcliffe Efficiency: {final_nse:.4f}")
print(f"Kling-Gupta Efficiency   : {final_kge:.4f}")


Step 1: Processing Static Attributes...
Successfully loaded static attributes for 671 basins.

Step 2: Processing Dynamic Forcings & Streamflow Targets...
 -> Loading & parsing files for Region 18/18...
Processing complete! Building global scales and tensors...
Successfully compiled data arrays for 671 active basins.

Experimental Split Configuration:
 -> Gauged Basins (Training): 637 basins from regions 1-17
 -> Ungauged Basins (Testing): 40 basins from hidden region 18

[Data] Generating Rolling Time-Windows...
Created dataset split containing 153896 sequence windows.
Created dataset split containing 9924 sequence windows.

>>> Dimensions Autodetected Successfully! <<<
Weather Features Input Matrix: 7
Static Attributes Vector Length: 53
Train windows: 153896 | Test windows: 9924 | Batch size: 64
Batches per epoch: 2404 (was 47979 before the change)
Model successfully loaded onto cuda.
Total trainable parameters: 201,729

--- Starting Training Phase on Gauged Basins ---
Epoch [1/10] C